<img src="images/logodwengo.png" alt="LogoDwengo" width="150"/>

<div>
    <font color=#690027 markdown="1">
<h1>SIMULATE AN EPIDEMIC: THE SIR MODEL</h1>    </font>
</div>

<div class="alert alert-box alert-success">
In this project, you will study how diseases can spread through a (social) network. You will investigate how the structure of a network can influence how quickly a disease is transmitted. Finally, you will also examine various strategies to combat the spread of a disease.<br>In this notebook, you will become familiar with the mathematical model SIR.</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.spatial import distance_matrix

## The SIR-model
One of the simplest ways to model disease spread in a community is using the SIR model. **SIR stands for *Susceptible* (vulnerable), *Infected* (infected), and *Resistant* (resistant or recovered), the three types of individuals found in a community.** <br>The SIR model consists of three equations that describe the changes in the number of individuals in a certain group. The variables that describe the state are:
- $S(t)$: the number of susceptible individuals at time $t$;-  $I(t)$: the number of infected individuals at time $t$;-  $R(t)$: the number of resistant individuals at time $t$.
Here t is the time in a certain unit of time (the unit of time is chosen depending on the problem).
This description makes a first major simplification of reality. It is assumed that each of these variables is a real number, and that the number of individuals in each group can vary continuously. In reality, these are discrete values: the number of infected and susceptible individuals is a natural number, after all, you are either infected or you are not. However, modelers prefer working with continuous variables because then they can use the techniques of mathematical analysis.

**Exercise 1**: Under what circumstances does this continuous approximation approximately apply? Do you think you can use this model to describe a family of four people?

Answer:

These three variables are linked to each other by means of three differential equations (each describing a change in time). It is assumed that the size of the population remains unchanged: so you assume that, during the time span described by the model, no one is born and no one dies. Essentially, you limit yourself here to the spread of a relatively harmless disease like a cold. So you can represent the situation with the following system of differential equations: 
The input doesn't seem to contain any text to translate.\left\{    \begin{array}{31}\Large\frac{\text{d}S(t)}{\text{d}t} \normalsize = -\beta \, S(t) \, I(t) \\\Large\frac{\text{d}I(t)}{\text{d}t} \normalsize = \beta \, S(t) \,I(t) - \gamma \, I(t) \\        \Large\frac{\text{d}R(t)}{\text{d}t} \normalsize = \gamma \, I(t)    \end{array}The input text doesn't seem to contain Dutch language text, HTML or markdown syntax, or Python comments. Therefore, the input is returned as it is: \right.$$

<div class="alert alert-box alert-info">
Each equation shows how the number of people in each group changes over time. From this, you can calculate how many people are in each group at a given time. The parameters $\beta$ and $\gamma$ play a fundamental role in this.</div>

The equations are linked through the *transition rates* (see figure). Each transition rate tells how to move from one group to another. <br>The transition rate from susceptible (S) to infected (I) depends on the contact between a susceptible person and an infected person. This is called *infection rate* $\beta$. This means that one infected person will infect $\beta S$ people. So, the number of susceptible people decreases by $\beta S I$ per unit of time. <br>The transition rate from infected (I) to resistant (R) only depends on the *recovery rate*, which is denoted as $\gamma$. Therefore, the number of infected individuals decreases by $\gamma I$ per unit of time.<img src="images/overgangSIR.png" alt="overgang in SIR" width="400"/>
<center>Figure 1: Transition from one group to another within the SIR model.</center>

**Exercise 2**: Write down how the numbers within each group change per unit of time.

Answer:

<div class="alert alert-box alert-info">
The SIR model is difficult to solve exactly. This is the case with many differential equations that occur in biological sciences. Thus, you must find a <em>numerical approximation</em> of the solution. This means that you will use an algorithm to find an estimated but accurate solution. From this solution, you can learn how the different variables change over time.</div>

There are several ways to do this:
- You could replace the continuous problem with a **discrete** counterpart. <br>This would allow you to use certain numerical methods to obtain an approximate solution.
- On the other hand, you can use an **iterative** method. <br>Starting from an initial estimate, iterative methods make successive approximations that gradually converge to the exact solution.

## Iterative way
With the help of computers, it is easy to find an iterative numerical solution to the SIR model.
- To do this, you start with an *initial condition*: it makes sense to start with a population with zero resistant persons, a few infected persons, and the rest susceptible (see examples).- Next, you can use the numerical solution to calculate the number of people in each group at certain times.
You can easily *simulate* such differential equations using the Python module SciPy.- First, you must *implement* the differential equations: to do this, put the three equations given above into a *row matrix*.<br> With the help of the Python module NumPy, a matrix can be input with a *NumPy array*.

In [ ]:
# ingeven differentiaalvergelijkingen
def SIR(t, y, beta, gamma):
    """Differentiaalvergelijkingen die S, I en R in functie van de tijd t bepalen."""
    S, I, R = y
    return np.array([-beta * S * I,
                    beta * S * I - gamma * I,
                    gamma * I])

- Now you can *numerically solve* the system of differential equations with the `solve_ivp()` function from the SciPy module for a certain *initial situation*.

### Example 1Consider a population with 1000 people, of which initially one person ($I_0$) is infected and $S_0=999$ people are susceptible to the disease.<br>You also provide the transition rates: $ \beta = 0,001$ and $\gamma = 0,1$.

In [ ]:
# voorbeeld 1 
# beginsituatie
S0 = 999
I0 = 1
R0 = 0

beta = 0.001
gamma = 0.1

oplossing = solve_ivp(SIR,                     # functie met parameters
                      [0, 100],                # tijdsinterval waarin je simuleert
                      np.array([S0, I0, R0]),  # initiële omstandigheden
                      args=(beta, gamma))      # parameters van stelsel differentiaalvergelijkingen 

In [ ]:
print(oplossing)       # oplossing geeft rij t-waarden en matrix y met als rijen S, I en R terug  

You then visually display this solution in several ways:

In [ ]:
# voorbeeld 1 grafiek oplossing S, I, R 
plt.figure()

plt.plot(oplossing.t, oplossing.y[0], color="orange")  # S
plt.plot(oplossing.t, oplossing.y[1], color="purple")  # I 
plt.plot(oplossing.t, oplossing.y[2], color="green")   # R

plt.show()

In [ ]:
# voorbeeld 1 grafiek verdeling populatie over S, I, R in functie van de tijd
plt.figure()

plt.stackplot(oplossing.t, oplossing.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "lightgreen"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.show()

In [ ]:
# grafiek voorbeeld 1 combinatie verdeling populatie en S, I, R
plt.figure()

plt.stackplot(oplossing.t, oplossing.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "lightgreen"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.plot(oplossing.t, oplossing.y[1], color="purple")  # I 
plt.plot(oplossing.t, oplossing.y[2], color="green")   # R
plt.plot(oplossing.t, oplossing.y[0], color="orange")  # S

plt.show()

> **Exercise 3**: Adjust example 1 and simulate a number of situations by adjusting the parameters:- What if initially half of the population was resistant?- What if initially 80% of the population was resistant?

### Example 2Same problem as in example 2 but with different transition percentages.

In [ ]:
# voorbeeld 2 
# beginsituatie
S0 = 999
I0 = 1
R0 = 0

beta = 0.0001
gamma = 0.048


oplossing4 = solve_ivp(SIR,                    # functie met parameters
                      [0, 365],                # tijdsinterval waarin we simuleren
                      np.array([S0, I0, R0]),  # initiële omstandigheden
                      t_eval=np.linspace(0,365,36),   # aantal punten van oplossing
                      args=(beta, gamma))      # parameters van stelsel differentiaalvergelijkingen 

In [ ]:
# voorbeeld 2 grafiek oplossing S, I, R 
plt.figure()

plt.plot(oplossing4.t, oplossing4.y[0], color="orange")  # S
plt.plot(oplossing4.t, oplossing4.y[1], color="purple")  # I 
plt.plot(oplossing4.t, oplossing4.y[2], color="green")   # R

plt.show()

In [ ]:
# grafiek voorbeeld 2 grafiek verdeling populatie over S, I, R in functie van de tijd
plt.figure()

plt.stackplot(oplossing4.t, oplossing4.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "green"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.show()

In [ ]:
# grafiek voorbeeld 2 combinatie verdeling populatie en S, I, R
plt.figure()

plt.stackplot(oplossing4.t, oplossing4.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "lightgreen"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.plot(oplossing.t, oplossing.y[1], color="purple")  # I 
plt.plot(oplossing.t, oplossing.y[2], color="green")   # R
plt.plot(oplossing.t, oplossing.y[0], color="orange")  # S

plt.show()

**Exercise 4**: Calculate the value of $S$ at the time when $I$ is at its maximum.

Answer:

**Exercise 5**: Modify Example 2 by reducing the infection rate $\beta$ by a quarter. How does the graph change?

<img src="images/cclic.png" alt="Banner" align="left" width="100"/><br><br>
This notebook by M. Stock and F. Wyffels for Dwengo vzw is licensed under a <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License</a>.